# Tropical Pacific MBL CO₂ — Full Analysis Notebook
**Author:** GML  
**Project:** Multi-platform comparison of tropical marine boundary layer CO₂ against model reference products  
**Period:** 2016–2018  
**Platforms:** TF5 Picarro (ship), POC Flask (ship), ATom DC-8 (aircraft)  
**References:** NOAA Global MBL, NOAA Pacific MBL, CarbonTracker CT2022  
**Remote Sensing:** OISST SST, MODIS Chlorophyll-a, ERA5 Wind + Pressure

## 0. Setup — Imports and Directory Structure
This cell sets up all imports and creates the output directories automatically. All paths are defined here so they only need to be changed in one place.

In [ ]:
import numpy as np
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import os
import urllib.request
import imageio
from scipy.interpolate import griddata
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.inspection import PartialDependenceDisplay

# ── Base paths ────────────────────────────────────────────────────────────────
BASE       = '/Volumes/USBDRIVE/finalproj/'

# ── Input data paths (pre-existing, downloaded separately) ───────────────────
SHIP_DIR   = BASE + 'data/observations/ship/'
ATOM_DIR   = BASE + 'data/observations/atom/'
MBL_GLOBAL = BASE + 'data/mbl/mbl_global/surface.mbl.co2'
MBL_PACIFIC= BASE + 'data/mbl/mbl_pacific/surface.mbl.co2'
MBL_GHG    = BASE + 'data/mbl/co2_GHGreference.1077328371_surface.txt'

# ── Remote sensing paths (auto-downloaded below) ─────────────────────────────
RS_DIR     = BASE + 'data/remote_sensing/'
SST_PATH   = RS_DIR + 'sst_oisst_tropics_monthly_2016_2018.nc'
CHL_FILES  = [RS_DIR + f'chl_modis_tropics_{y}.nc' for y in [2016, 2017, 2018]]
ERA5_PATH  = RS_DIR + 'era5_slp_wind_tropics_2016_2018.nc'
CT_DIR     = RS_DIR + 'carbontracker/'

# ── Output paths ──────────────────────────────────────────────────────────────
FIG_DIR    = BASE + 'outputs/figures/'
GIF_DIR    = BASE + 'outputs/gifs/'
FRAME_DIR  = BASE + 'outputs/frames/'
CSV_DIR    = BASE + 'outputs/csv/'

# ── Create all directories ────────────────────────────────────────────────────
for d in [SHIP_DIR, ATOM_DIR, os.path.dirname(MBL_GLOBAL),
          RS_DIR, CT_DIR, FIG_DIR, GIF_DIR, FRAME_DIR, CSV_DIR]:
    os.makedirs(d, exist_ok=True)

print('All directories ready.')
print(f'Figures  → {FIG_DIR}')
print(f'GIFs     → {GIF_DIR}')
print(f'CSVs     → {CSV_DIR}')

## 1. Download Remote Sensing and Model Data
This cell downloads SST, chlorophyll, ERA5, and CarbonTracker automatically. Ship and ATom data need to be placed manually in their respective directories (see paths above). Each download is skipped if the file already exists.

In [ ]:
# ── Download SST (NOAA OISST monthly, tropics 2016-2018) ──────────────────────
if not os.path.exists(SST_PATH):
    print('Downloading SST...')
    url = (
        'https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst2Agg_LonPM180.nc'
        '?sst[(2016-01-16T12:00:00Z):30:(2018-12-16T12:00:00Z)]'
        '[(0.0):1:(0.0)][(-23.5):1:(23.5)][(-180.0):1:(180.0)]'
    )
    urllib.request.urlretrieve(url, SST_PATH)
    print(f'  Saved to {SST_PATH}')
else:
    print('SST already downloaded.')

# ── Download Chlorophyll (MODIS-Aqua monthly, tropics, one file per year) ─────
for year, chl_path in zip([2016, 2017, 2018], CHL_FILES):
    if not os.path.exists(chl_path):
        print(f'Downloading chlorophyll {year}...')
        url = (
            f'https://coastwatch.pfeg.noaa.gov/erddap/griddap/erdMH1chlamday.nc'
            f'?chlorophyll[({year}-01-16T00:00:00Z):1:({year}-12-16T00:00:00Z)]'
            f'[(-23.5):8:(23.5)][(-180.0):8:(180.0)]'
        )
        urllib.request.urlretrieve(url, chl_path)
        print(f'  Saved to {chl_path}')
    else:
        print(f'Chlorophyll {year} already downloaded.')

# ── Download ERA5 (MSL pressure + wind, monthly, tropics 2016-2018) ───────────
if not os.path.exists(ERA5_PATH):
    print('Downloading ERA5 (requires CDS API key in ~/.cdsapirc)...')
    import cdsapi
    c = cdsapi.Client()
    c.retrieve(
        'reanalysis-era5-single-levels-monthly-means',
        {
            'product_type': 'monthly_averaged_reanalysis',
            'variable': ['mean_sea_level_pressure',
                         '10m_u_component_of_wind',
                         '10m_v_component_of_wind'],
            'year':  ['2016', '2017', '2018'],
            'month': ['01','02','03','04','05','06',
                      '07','08','09','10','11','12'],
            'time':  '00:00',
            'area':  [23.5, -180, -23.5, 180],
            'format': 'netcdf',
        },
        ERA5_PATH
    )
    print(f'  Saved to {ERA5_PATH}')
else:
    print('ERA5 already downloaded.')

# ── Download CarbonTracker CT2022 (monthly, 2016-2018) ────────────────────────
ct_base = 'https://gml.noaa.gov/aftp/products/carbontracker/co2/molefractions/co2_total_monthly/'
for year in [2016, 2017, 2018]:
    for month in range(1, 13):
        fname = f'CT2022.molefrac_glb3x2_{year}-{month:02d}.nc'
        fpath = CT_DIR + fname
        if not os.path.exists(fpath):
            print(f'  Downloading {fname}...')
            try:
                urllib.request.urlretrieve(ct_base + fname, fpath)
            except Exception as e:
                print(f'  ERROR: {e}')

print('All downloads complete.')

## 2. Load Ship Observations
This cell reads both ship datasets from NetCDF. The TF5 is a Picarro continuous analyzer at 1 Hz, the POC is a NOAA GML flask sampler. Both are filtered to the tropics (23.5S-23.5N) and 2016-2018. The data is stored as mol/mol so we convert to ppm by multiplying by 1e6.

In [ ]:
POC_FILE = SHIP_DIR + 'co2_poc_shipboard-flask_1_representative.nc'
TF5_FILE = SHIP_DIR + 'co2_tf5_shipboard-insitu_20_allvalid.nc'

DATE_START = pd.Timestamp('2016-01-01')
DATE_END   = pd.Timestamp('2018-12-31')
LAT_MIN, LAT_MAX = -23.5, 23.5

def decode_time(ds):
    return nc.num2date(
        ds.variables['time'][:],
        units=ds.variables['time'].units,
        calendar=getattr(ds.variables['time'], 'calendar', 'standard')
    )

def get_co2_var(ds):
    for name in ['xco2_dry', 'co2_dry', 'xco2', 'value', 'co2']:
        if name in ds.variables:
            print(f'  using variable: {name}')
            return name
    raise ValueError('No CO2 variable found')

datasets = {}
for label, fpath in [('POC flask', POC_FILE), ('TF5 Picarro', TF5_FILE)]:
    print(f'Loading {label}...')
    with nc.Dataset(fpath) as ds:
        times_raw = decode_time(ds)
        times_dt  = np.array([pd.Timestamp(t.year, t.month, t.day,
                                            t.hour, t.minute, t.second)
                               for t in times_raw])
        lat = ds.variables['latitude'][:].data.flatten()
        lon = ds.variables['longitude'][:].data.flatten()
        co2_var = get_co2_var(ds)
        co2_raw = np.ma.filled(ds.variables[co2_var][:], np.nan).flatten()
        co2_raw = co2_raw * 1e6  # mol/mol -> ppm

    time_mask = (times_dt >= DATE_START) & (times_dt <= DATE_END)
    lat_mask  = (lat >= LAT_MIN) & (lat <= LAT_MAX)
    mask      = time_mask & lat_mask & np.isfinite(co2_raw) & (co2_raw > 350) & (co2_raw < 450)

    datasets[label] = dict(time=times_dt[mask], lat=lat[mask],
                            lon=lon[mask], co2=co2_raw[mask])
    print(f'  {mask.sum()} points after filtering')

## 3. Load ATom Aircraft Data
This cell reads all ATom MER-1HZ ICARTT files from the ATom directory. ICARTT is a NASA text format where the first number on line 1 tells you how many header lines to skip. We parse the date from the filename, convert UTC_Start seconds to datetimes, and filter to the tropical MBL (below 500m altitude).

In [ ]:
def read_icartt(filepath):
    with open(filepath, 'r', errors='replace') as f:
        first_line = f.readline().strip()
    n_header = int(first_line.split(',')[0])
    df = pd.read_csv(filepath, skiprows=n_header-1, sep=',',
                     na_values=['-99999','-99999.0','-9999','-7777','-8888'],
                     low_memory=False)
    df.columns = df.columns.str.strip()
    return df

def date_from_filename(fname):
    for p in os.path.basename(fname).split('_'):
        if len(p) == 8 and p.isdigit():
            return pd.Timestamp(p)
    return None

files = sorted(glob.glob(ATOM_DIR + 'MER-1HZ_DC8_*.ict'))
print(f'Found {len(files)} ATom files')

atom_chunks = []
for fpath in files:
    flight_date = date_from_filename(fpath)
    try:
        df_raw = read_icartt(fpath)
    except Exception as e:
        print(f'  ERROR {os.path.basename(fpath)}: {e}')
        continue

    co2_col = 'CO2_NOAA' if 'CO2_NOAA' in df_raw.columns else \
              'CO2_QCLS' if 'CO2_QCLS' in df_raw.columns else None
    if co2_col is None:
        continue

    df_raw['datetime'] = flight_date + pd.to_timedelta(df_raw['UTC_Start'], unit='s')
    chunk = df_raw[['datetime','G_LAT','G_LONG','G_ALT', co2_col]].copy()
    chunk.rename(columns={'G_LAT':'lat','G_LONG':'lon',
                           'G_ALT':'alt', co2_col:'co2'}, inplace=True)

    mask = ((chunk['lat'] >= LAT_MIN) & (chunk['lat'] <= LAT_MAX) &
            (chunk['alt'] >= 0) & (chunk['alt'] <= 500) &
            chunk['co2'].notna() & (chunk['co2'] > 350) & (chunk['co2'] < 450))
    filtered = chunk[mask].copy()
    if len(filtered) > 0:
        atom_chunks.append(filtered)
        print(f'  {os.path.basename(fpath)}: {len(filtered)} MBL tropical points')

atom = pd.concat(atom_chunks, ignore_index=True)
atom['lon'] = ((atom['lon'] + 180) % 360) - 180

def assign_campaign(dt):
    if   pd.Timestamp('2016-07-01') <= dt <= pd.Timestamp('2016-10-01'): return 'ATom-1'
    elif pd.Timestamp('2016-11-01') <= dt <= pd.Timestamp('2017-03-01'): return 'ATom-2'
    elif pd.Timestamp('2017-08-01') <= dt <= pd.Timestamp('2017-11-01'): return 'ATom-3'
    elif pd.Timestamp('2018-03-01') <= dt <= pd.Timestamp('2018-06-01'): return 'ATom-4'
    else: return 'Other'

atom['campaign'] = atom['datetime'].apply(assign_campaign)
atom['alt_col']  = atom['alt']
print(f'Total ATom MBL tropical points: {len(atom)}')

## 4. Combine All Platforms and Save CSV
This cell combines ship and aircraft data into one DataFrame with a source column. Ship altitude is set to 0 (surface). We save this to a CSV so we only need to load raw files once.

In [ ]:
ship_chunks = []
for label, d in datasets.items():
    df_ship = pd.DataFrame({'datetime': d['time'], 'lat': d['lat'],
                             'lon': d['lon'], 'alt': 0.0,
                             'co2': d['co2'], 'source': label})
    ship_chunks.append(df_ship)

ships_combined = pd.concat(ship_chunks, ignore_index=True)
atom_out = atom[['datetime','lat','lon','alt_col','co2','campaign']].copy()
atom_out.rename(columns={'campaign':'source','alt_col':'alt'}, inplace=True)

combined = pd.concat([ships_combined, atom_out], ignore_index=True)
combined.sort_values('datetime', inplace=True)
combined.reset_index(drop=True, inplace=True)

CSV_PATH = CSV_DIR + 'tropical_mbl_co2_combined.csv'
combined.to_csv(CSV_PATH, index=False)
print(f'Saved {len(combined)} points to {CSV_PATH}')
print(combined['source'].value_counts())

## 5. Load CSV and Compute MBL Offsets
This cell loads the combined CSV and snaps each observation to the three reference products. We convert datetime to decimal year and latitude to sine latitude to match the MBL grid coordinate system. The offset is computed as obs minus model — positive means observations are higher than the reference.

In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=['datetime'])
print(f'Loaded {len(df)} observations')

def to_decimal_year(dt):
    year  = dt.year
    start = pd.Timestamp(year, 1, 1)
    end   = pd.Timestamp(year + 1, 1, 1)
    return year + (dt - start) / (end - start)

df['dec_year'] = df['datetime'].apply(to_decimal_year)
df['sinlat']   = np.sin(np.deg2rad(df['lat']))
df['doy']      = df['datetime'].dt.dayofyear
df['lon_360']  = df['lon'] % 360
df['year_month'] = df['datetime'].dt.to_period('M')

# MBL reader
sinlat_bands_mbl = np.linspace(-1.0, 1.0, 41)

def load_mbl(fpath):
    rows = []
    with open(fpath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            parts = line.split()
            if len(parts) < 5: continue
            try: float(parts[0])
            except ValueError: continue
            rows.append([float(p) for p in parts])
    rows = np.array(rows)
    return rows[:, 0], rows[:, 1:]

def snap_mbl(dec_year, sinlat, mbl_time, mbl_vals):
    t_idx = np.argmin(np.abs(mbl_time         - dec_year))
    l_idx = np.argmin(np.abs(sinlat_bands_mbl - sinlat))
    return mbl_vals[t_idx, l_idx]

# GHG reference (13 bands)
sinlat_bands_ghg = np.array([-0.30,-0.25,-0.20,-0.15,-0.10,
                               -0.05, 0.00, 0.05, 0.10, 0.15,
                                0.20, 0.25, 0.30])
val_col_indices  = [1,3,5,7,9,11,13,15,17,19,21,23,25]
rows_ghg = []
with open(MBL_GHG, 'r') as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'): continue
        parts = line.split()
        if len(parts) < 2: continue
        try: float(parts[0])
        except ValueError: continue
        rows_ghg.append([float(p) for p in parts])
rows_ghg  = np.array(rows_ghg)
ghg_time  = rows_ghg[:, 0]
ghg_vals  = rows_ghg[:, val_col_indices]

def snap_ghg(dec_year, sinlat):
    t_idx = np.argmin(np.abs(ghg_time          - dec_year))
    l_idx = np.argmin(np.abs(sinlat_bands_ghg  - sinlat))
    return ghg_vals[t_idx, l_idx]

print('Loading Global MBL...')
glob_time, glob_vals = load_mbl(MBL_GLOBAL)
print('Loading Pacific MBL...')
pac_time,  pac_vals  = load_mbl(MBL_PACIFIC)

print('Snapping to references...')
df['ghg_mbl']     = [snap_ghg(r.dec_year, r.sinlat) for r in df.itertuples()]
df['global_mbl']  = [snap_mbl(r.dec_year, r.sinlat, glob_time, glob_vals) for r in df.itertuples()]
df['pacific_mbl'] = [snap_mbl(r.dec_year, r.sinlat, pac_time,  pac_vals)  for r in df.itertuples()]
df['offset']         = df['co2'] - df['ghg_mbl']
df['offset_global']  = df['co2'] - df['global_mbl']
df['offset_pacific'] = df['co2'] - df['pacific_mbl']
print('Done.')

## 6. Load CarbonTracker
This cell loads all 36 CarbonTracker monthly files and stacks the pbl_co2 field into a single (time, lat, lon) array. pbl_co2 is the CO2 concentration averaged through the planetary boundary layer — the most appropriate field for comparing to surface and low-altitude observations.

In [ ]:
ct_files = sorted(glob.glob(CT_DIR + 'CT2022.molefrac_glb3x2_*.nc'))
print(f'Found {len(ct_files)} CarbonTracker files')

ct_times, ct_pbl = [], []
ct_lats = ct_lons = None
for fpath in ct_files:
    with nc.Dataset(fpath) as ds:
        ct_times.append(float(ds.variables['decimal_date'][0]))
        if ct_lats is None:
            ct_lats = ds.variables['latitude'][:]
            ct_lons = ds.variables['longitude'][:]
        ct_pbl.append(np.ma.filled(ds.variables['pbl_co2'][0], np.nan))

ct_times = np.array(ct_times)
ct_pbl   = np.array(ct_pbl)

def snap_ct(dec_year, obs_lat, obs_lon):
    obs_lon_360 = obs_lon % 360
    t_idx = np.argmin(np.abs(ct_times - dec_year))
    l_idx = np.argmin(np.abs(ct_lats  - obs_lat))
    o_idx = np.argmin(np.abs(ct_lons  - obs_lon_360))
    return ct_pbl[t_idx, l_idx, o_idx]

print('Snapping to CarbonTracker...')
ct_matched = []
for i, r in enumerate(df.itertuples()):
    ct_matched.append(snap_ct(r.dec_year, r.lat, r.lon))
    if i % 10000 == 0: print(f'  {i}/{len(df)}...')
df['ct_pbl']    = ct_matched
df['offset_ct'] = df['co2'] - df['ct_pbl']
print('Done.')

## 7. Load and Match Remote Sensing Data
This cell loads SST, chlorophyll, and ERA5 and snaps each observation to the nearest grid cell and monthly time step. We use a simple nearest-neighbor approach. ERA5 longitude is 0-360 so we convert observation longitudes before matching.

In [ ]:
# ── SST ───────────────────────────────────────────────────────────────────────
print('Loading SST...')
with nc.Dataset(SST_PATH) as ds:
    sst_time_raw = nc.num2date(ds.variables['time'][:],
                               units=ds.variables['time'].units,
                               calendar=getattr(ds.variables['time'],'calendar','standard'))
    sst_time = pd.to_datetime([pd.Timestamp(t.year,t.month,t.day) for t in sst_time_raw])
    sst_lat  = ds.variables['latitude'][:]
    sst_lon  = ds.variables['longitude'][:]
    sst_data = np.ma.filled(ds.variables['sst'][:].squeeze(), np.nan)

def get_sst(obs_time, obs_lat, obs_lon):
    t_idx = np.argmin(np.abs(sst_time - obs_time))
    l_idx = np.argmin(np.abs(sst_lat  - obs_lat))
    o_idx = np.argmin(np.abs(sst_lon  - obs_lon))
    return sst_data[t_idx, l_idx, o_idx]

print('Matching SST...')
sst_matched = []
for i, r in enumerate(df.itertuples()):
    sst_matched.append(get_sst(r.datetime, r.lat, r.lon))
    if i % 10000 == 0: print(f'  {i}/{len(df)}...')
df['sst'] = sst_matched

# ── Chlorophyll ───────────────────────────────────────────────────────────────
print('Loading chlorophyll...')
chl_time_list, chl_data_list = [], []
for fpath in CHL_FILES:
    with nc.Dataset(fpath) as ds:
        t_raw = nc.num2date(ds.variables['time'][:],
                            units=ds.variables['time'].units,
                            calendar=getattr(ds.variables['time'],'calendar','standard'))
        chl_time_list.extend([pd.Timestamp(t.year,t.month,t.day) for t in t_raw])
        data = np.ma.filled(ds.variables['chlorophyll'][:].squeeze(), np.nan)
        if data.ndim == 2: data = data[np.newaxis,:,:]
        chl_data_list.append(data)
        chl_lat = ds.variables['latitude'][:]
        chl_lon = ds.variables['longitude'][:]

chl_time = pd.to_datetime(chl_time_list)
chl_data = np.concatenate(chl_data_list, axis=0)

def get_chl(obs_time, obs_lat, obs_lon):
    t_idx = np.argmin(np.abs(chl_time - obs_time))
    l_idx = np.argmin(np.abs(chl_lat  - obs_lat))
    o_idx = np.argmin(np.abs(chl_lon  - obs_lon))
    return chl_data[t_idx, l_idx, o_idx]

print('Matching chlorophyll...')
chl_matched = []
for i, r in enumerate(df.itertuples()):
    chl_matched.append(get_chl(r.datetime, r.lat, r.lon))
    if i % 10000 == 0: print(f'  {i}/{len(df)}...')
df['chl'] = chl_matched
df['log_chl'] = np.log10(df['chl'].clip(lower=1e-4))

# ── ERA5 ──────────────────────────────────────────────────────────────────────
print('Loading ERA5...')
with nc.Dataset(ERA5_PATH) as ds:
    era5_time_raw = nc.num2date(ds.variables['valid_time'][:],
                                units=ds.variables['valid_time'].units,
                                calendar='standard')
    era5_time = pd.to_datetime([pd.Timestamp(t.year,t.month,t.day) for t in era5_time_raw])
    era5_lat  = ds.variables['latitude'][:]
    era5_lon  = ds.variables['longitude'][:]
    msl       = np.ma.filled(ds.variables['msl'][:], np.nan) / 100.0
    u10       = np.ma.filled(ds.variables['u10'][:], np.nan)
    v10       = np.ma.filled(ds.variables['v10'][:], np.nan)
    wspd      = np.sqrt(u10**2 + v10**2)

# sort era5 lon for plotting
era5_lon_pm180 = np.where(era5_lon > 180, era5_lon - 360, era5_lon)
sort_idx = np.argsort(era5_lon_pm180)
era5_lon_sorted = era5_lon_pm180[sort_idx]

def get_era5(obs_time, obs_lat, obs_lon, field):
    obs_lon_360 = obs_lon % 360
    t_idx = np.argmin(np.abs(era5_time - obs_time))
    l_idx = np.argmin(np.abs(era5_lat  - obs_lat))
    o_idx = np.argmin(np.abs(era5_lon  - obs_lon_360))
    return field[t_idx, l_idx, o_idx]

print('Matching ERA5...')
msl_matched, wspd_matched = [], []
for i, r in enumerate(df.itertuples()):
    msl_matched.append(get_era5(r.datetime,  r.lat, r.lon, msl))
    wspd_matched.append(get_era5(r.datetime, r.lat, r.lon, wspd))
    if i % 10000 == 0: print(f'  {i}/{len(df)}...')
df['msl']  = msl_matched
df['wspd'] = wspd_matched

# ── Save fully matched dataset ─────────────────────────────────────────────────
FULL_CSV = CSV_DIR + 'tropical_mbl_co2_fully_matched.csv'
df.to_csv(FULL_CSV, index=False)
print(f'Saved fully matched dataset to {FULL_CSV}')

## 8. Shared Style Definitions
This cell defines the platform colors and product labels used across all plots so everything stays consistent.

In [ ]:
platform_style = {
    'POC flask':   {'color':'#E05C2A','marker':'o','size':50, 'zorder':6,'label':'POC Shipboard Flask'},
    'TF5 Picarro': {'color':'#2A7AE0','marker':'o','size':20, 'zorder':5,'label':'TF5 Shipboard Picarro'},
    'ATom-1':      {'color':'#2ECC71','marker':'^','size':40, 'zorder':4,'label':'ATom-1 (Aug 2016)'},
    'ATom-2':      {'color':'#9B59B6','marker':'^','size':40, 'zorder':4,'label':'ATom-2 (Jan-Feb 2017)'},
    'ATom-3':      {'color':'#F39C12','marker':'^','size':40, 'zorder':4,'label':'ATom-3 (Sep-Oct 2017)'},
    'ATom-4':      {'color':'#E74C3C','marker':'^','size':40, 'zorder':4,'label':'ATom-4 (Apr-May 2018)'},
}
product_labels = {
    'offset_global':  'Global MBL',
    'offset_pacific': 'Pacific MBL',
    'offset_ct':      'CarbonTracker PBL',
}
product_colors = {
    'offset_global':  '#E05C2A',
    'offset_pacific': '#2A7AE0',
    'offset_ct':      '#2ECC71',
}
product_line_style = {
    'offset_global':  {'color':'#E05C2A','linestyle':'-',  'label':'Global MBL'},
    'offset_pacific': {'color':'#2A7AE0','linestyle':'--', 'label':'Pacific MBL'},
    'offset_ct':      {'color':'#2ECC71','linestyle':'-.', 'label':'CarbonTracker PBL'},
}
print('Styles defined.')

## 9. Observation Maps and Time Series
These plots show where the data is and what the CO2 looks like across platforms.

In [ ]:
# PLOT: Time series of CO2 — all platforms
fig, ax = plt.subplots(figsize=(14,6))
for source, style in platform_style.items():
    sub = df[df['source']==source].sort_values('datetime')
    if len(sub)==0: continue
    ax.scatter(sub['datetime'],sub['co2'],color=style['color'],marker=style['marker'],
               s=style['size'],alpha=0.7,linewidths=0,zorder=style['zorder'],label=style['label'])
ax.set_xlabel('Date',fontsize=12,fontweight='bold')
ax.set_ylabel('CO\u2082 (ppm)',fontsize=12,fontweight='bold')
ax.set_title('Tropical MBL CO\u2082 \u2014 All Platforms (23.5\u00b0S\u201323.5\u00b0N)',fontsize=13,fontweight='bold')
ax.set_ylim(395,420)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(),rotation=30,ha='right')
ax.legend(fontsize=11,loc='upper left',bbox_to_anchor=(1.01,1),borderaxespad=0,framealpha=0.9)
ax.grid(True,alpha=0.3)
fig.tight_layout(rect=[0,0,0.82,1])
plt.savefig(FIG_DIR+'combined_co2_timeseries.png',dpi=150,bbox_inches='tight')
plt.show()

# PLOT: Globe map coloured by platform
fig = plt.figure(figsize=(10,10))
ax  = fig.add_subplot(1,1,1,projection=ccrs.Orthographic(central_longitude=180,central_latitude=0))
ax.set_global()
ax.add_feature(cfeature.OCEAN,facecolor='#C8E6F5',zorder=0)
ax.add_feature(cfeature.LAND,facecolor='#D6CFC4',zorder=1)
ax.add_feature(cfeature.COASTLINE,linewidth=0.6,zorder=2)
ax.gridlines(draw_labels=False,linewidth=0.5,color='grey',alpha=0.5,linestyle='--')
for source,style in platform_style.items():
    sub=df[df['source']==source]
    if len(sub)==0: continue
    ax.scatter(sub['lon'],sub['lat'],color=style['color'],marker=style['marker'],
               s=style['size'],alpha=0.75,linewidths=0,transform=ccrs.PlateCarree(),
               zorder=style['zorder'],label=style['label'])
for lat_line in [-23.5,23.5]:
    ax.plot(np.linspace(-180,180,500),np.full(500,lat_line),
            transform=ccrs.PlateCarree(),color='gold',linewidth=1.2,linestyle='--',zorder=7)
ax.legend(loc='lower left',fontsize=10,facecolor='white',framealpha=0.9)
ax.set_title('Tropical MBL CO\u2082 Observations \u2014 All Platforms',fontsize=13,fontweight='bold',pad=14)
plt.savefig(FIG_DIR+'combined_map_byplatform.png',dpi=150,bbox_inches='tight')
plt.show()

# PLOT: Globe map coloured by CO2 value
fig = plt.figure(figsize=(10,10))
ax  = fig.add_subplot(1,1,1,projection=ccrs.Orthographic(central_longitude=180,central_latitude=0))
ax.set_global()
ax.add_feature(cfeature.OCEAN,facecolor='#C8E6F5',zorder=0)
ax.add_feature(cfeature.LAND,facecolor='#D6CFC4',zorder=1)
ax.add_feature(cfeature.COASTLINE,linewidth=0.6,zorder=2)
ax.gridlines(draw_labels=False,linewidth=0.5,color='grey',alpha=0.5,linestyle='--')
ships = df[df['source'].isin(['POC flask','TF5 Picarro'])]
atom_df = df[df['source'].isin(['ATom-1','ATom-2','ATom-3','ATom-4'])]
sc = ax.scatter(atom_df['lon'],atom_df['lat'],c=atom_df['co2'],cmap='plasma',
                vmin=395,vmax=418,s=15,alpha=1.0,linewidths=0,transform=ccrs.PlateCarree(),zorder=4)
ax.scatter(ships['lon'],ships['lat'],c=ships['co2'],cmap='plasma',
           vmin=395,vmax=418,s=50,alpha=1.0,linewidths=0,transform=ccrs.PlateCarree(),zorder=5)
for lat_line in [-23.5,23.5]:
    ax.plot(np.linspace(-180,180,500),np.full(500,lat_line),
            transform=ccrs.PlateCarree(),color='gold',linewidth=1.2,linestyle='--',zorder=6)
cbar=plt.colorbar(sc,ax=ax,orientation='horizontal',pad=0.04,shrink=0.65,aspect=30)
cbar.set_label('CO\u2082 (ppm)',fontsize=11)
ax.set_title('Tropical MBL CO\u2082 \u2014 All Platforms\nColoured by CO\u2082 value',fontsize=13,fontweight='bold',pad=14)
plt.savefig(FIG_DIR+'combined_map_byco2.png',dpi=150,bbox_inches='tight')
plt.show()

## 10. MBL Offset Plots
These plots show the offset between observations and reference products. The offset is obs minus model — positive means observations are higher than the reference.

In [ ]:
# PLOT: Per-platform offset time series
sources = [s for s in platform_style.keys() if s in df['source'].values]
fig, axes = plt.subplots(len(sources),1,figsize=(14,3*len(sources)),sharex=True,sharey=False)
for ax, source in zip(axes,sources):
    style = platform_style[source]
    sub   = df[df['source']==source].sort_values('datetime')
    ax.axhline(0,color='black',linewidth=1.0,linestyle='--',zorder=2)
    ax.scatter(sub['datetime'],sub['offset'],color=style['color'],marker=style['marker'],
               s=style['size'],alpha=0.7,linewidths=0,zorder=3)
    mean_off = sub['offset'].mean()
    ax.annotate(f'mean = {mean_off:+.2f} ppm',xy=(0.98,0.88),xycoords='axes fraction',
                ha='right',fontsize=10,color=style['color'],fontweight='bold')
    ax.set_ylabel('Offset (ppm)',fontsize=10)
    ax.set_title(style['label'],fontsize=11,fontweight='bold',loc='left')
    ax.set_ylim(-5,5)
    ax.grid(True,alpha=0.3)
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(axes[-1].xaxis.get_majorticklabels(),rotation=30,ha='right')
fig.suptitle('CO\u2082 Offset from NOAA MBL Reference \u2014 By Platform',fontsize=13,fontweight='bold',y=1.01)
fig.tight_layout()
plt.savefig(FIG_DIR+'offset_by_platform.png',dpi=150,bbox_inches='tight')
plt.show()

# PLOT: Three-panel offset time series (one per product, all observations combined)
fig, axes = plt.subplots(3,1,figsize=(14,9),sharex=True,sharey=True)
for ax, col in zip(axes,['offset_global','offset_pacific','offset_ct']):
    ax.axhline(0,color='black',linewidth=1.2,linestyle='--',zorder=2)
    ax.scatter(df['datetime'],df[col],color='#065A82',s=8,alpha=0.6,linewidths=0,zorder=3)
    ax.set_ylabel('CO\u2082 offset (ppm)',fontsize=11,fontweight='bold')
    ax.set_title(f'Obs minus {product_labels[col]}',fontsize=12,fontweight='bold',loc='left')
    ax.set_ylim(-3,3)
    ax.grid(True,alpha=0.3)
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(axes[-1].xaxis.get_majorticklabels(),rotation=30,ha='right',fontsize=11)
fig.suptitle('CO\u2082 Observations vs Three Reference Products',fontsize=14,fontweight='bold')
fig.tight_layout()
plt.savefig(FIG_DIR+'offset_three_products_timeseries.png',dpi=150,bbox_inches='tight')
plt.show()

# PLOT: Globe maps of offsets (model minus obs), horizontal
fig, axes = plt.subplots(1,3,figsize=(18,7),
                          subplot_kw={'projection':ccrs.Orthographic(central_longitude=180,central_latitude=0)})
for ax, col in zip(axes,['offset_global','offset_pacific','offset_ct']):
    ax.set_global()
    ax.add_feature(cfeature.OCEAN,facecolor='#C8E6F5',zorder=0)
    ax.add_feature(cfeature.LAND,facecolor='#D6CFC4',zorder=1)
    ax.add_feature(cfeature.COASTLINE,linewidth=0.6,zorder=2)
    ax.gridlines(draw_labels=False,linewidth=0.5,color='grey',alpha=0.5,linestyle='--')
    valid = df[np.isfinite(df[col])]
    sc = ax.scatter(valid['lon'],valid['lat'],c=-valid[col],cmap='RdBu_r',
                    vmin=-3,vmax=3,s=20,alpha=1.0,linewidths=0,transform=ccrs.PlateCarree(),zorder=3)
    for lat_line in [-23.5,23.5]:
        ax.plot(np.linspace(-180,180,500),np.full(500,lat_line),
                transform=ccrs.PlateCarree(),color='gold',linewidth=1.2,linestyle='--',zorder=4)
    ax.set_title(product_labels[col],fontsize=12,fontweight='bold',pad=8)
fig.suptitle('CO\u2082 Offset Maps \u2014 Model minus Observations',fontsize=13,fontweight='bold',y=1.02)
fig.tight_layout(rect=[0,0.08,1,1])
cbar_ax=fig.add_axes([0.2,0.02,0.6,0.03])
sm=plt.cm.ScalarMappable(cmap='RdBu_r',norm=plt.Normalize(-3,3))
sm.set_array([])
cbar=plt.colorbar(sm,cax=cbar_ax,orientation='horizontal',extend='both')
cbar.set_label('CO\u2082 offset \u2014 Model minus Obs (ppm)',fontsize=11)
plt.savefig(FIG_DIR+'offset_three_products_maps.png',dpi=150,bbox_inches='tight')
plt.show()

## 11. Hovmöller Diagrams
These diagrams show the CO2 offset as a function of latitude (or longitude) and time. We bin observations into 2-degree latitude bins and monthly time steps, then interpolate using scipy griddata to produce smooth contour plots. The sign is model minus obs.

In [ ]:
time_bins    = sorted(df['year_month'].unique())
time_centers = [t.to_timestamp() for t in time_bins]

lat_bins    = np.arange(-23.5,24.5,2)
lat_centers = (lat_bins[:-1]+lat_bins[1:])/2

def make_hovmoller(df, products, bin_var, bin_bins, bin_centers,
                   xlabel, savename):
    fig, axes = plt.subplots(1,3,figsize=(18,8),sharey=True,sharex=True)
    for ax, col in zip(axes,products):
        pts_t,pts_l,pts_v=[],[],[]
        for ti,tb in enumerate(time_bins):
            sub_t=df[df['year_month']==tb]
            for li,(lo,hi) in enumerate(zip(bin_bins[:-1],bin_bins[1:])):
                chunk=sub_t[(sub_t[bin_var]>=lo)&(sub_t[bin_var]<hi)][col].dropna()
                if len(chunk)>=3:
                    pts_t.append(mdates.date2num(time_centers[ti]))
                    pts_l.append(bin_centers[li])
                    pts_v.append(-chunk.mean())
        if len(pts_t)<5:
            ax.text(0.5,0.5,'insufficient data',ha='center',va='center',transform=ax.transAxes)
            continue
        pts_t=np.array(pts_t); pts_l=np.array(pts_l); pts_v=np.array(pts_v)
        t_grid=np.linspace(pts_t.min(),pts_t.max(),200)
        l_grid=np.linspace(bin_centers.min(),bin_centers.max(),100)
        T,L=np.meshgrid(t_grid,l_grid)
        V=griddata((pts_t,pts_l),pts_v,(T,L),method='linear')
        ax.contourf(T,L,V,levels=np.linspace(-2,2,21),cmap='RdBu_r',extend='both')
        ax.contour(T,L,V,levels=[-1.5,-1,-0.5,0.5,1,1.5],colors='k',linewidths=0.4,alpha=0.4)
        ax.axhline(0,color='black',linewidth=0.8,linestyle='--',zorder=5)
        ax.set_title(product_labels[col],fontsize=11,fontweight='bold')
        ax.set_xlabel('Date',fontsize=11,fontweight='bold')
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7]))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        plt.setp(ax.xaxis.get_majorticklabels(),rotation=30,ha='right',fontsize=9)
        ax.grid(True,alpha=0.15)
    axes[0].set_ylabel(xlabel,fontsize=11,fontweight='bold')
    fig.suptitle(f'CO\u2082 Offset \u2014 Model minus Observations\n{xlabel} vs Time',
                 fontsize=13,fontweight='bold')
    fig.tight_layout(rect=[0,0.06,1,1])
    cbar_ax=fig.add_axes([0.2,0.02,0.6,0.025])
    sm=plt.cm.ScalarMappable(cmap='RdBu_r',norm=plt.Normalize(-2,2))
    sm.set_array([])
    cbar=plt.colorbar(sm,cax=cbar_ax,orientation='horizontal',extend='both')
    cbar.set_label('CO\u2082 offset \u2014 Model minus Obs (ppm)',fontsize=11)
    plt.savefig(FIG_DIR+savename,dpi=150,bbox_inches='tight')
    plt.show()

# Latitude Hovmöller
make_hovmoller(df,['offset_global','offset_pacific','offset_ct'],
               'lat',lat_bins,lat_centers,'Latitude (\u00b0)','hovmoller_latitude.png')

# Longitude Hovmöller
lon_bins    = np.arange(0,365,10)
lon_centers = (lon_bins[:-1]+lon_bins[1:])/2
make_hovmoller(df,['offset_global','offset_pacific','offset_ct'],
               'lon_360',lon_bins,lon_centers,'Longitude (\u00b0)','hovmoller_longitude.png')

## 12. Remote Sensing Scatter Plots
These scatter plots show CO2 and CO2 offset against the four remote sensing variables. All one color for clarity — the red trend line shows the linear fit. We also plot raw CO2 (not just offset) against each variable.

In [ ]:
def scatter_two(xvar, xlabel, co2_ylim=(395,420), offset_ylim=(-5,5),
                xdf=None, save_prefix='scatter'):
    xdf = xdf if xdf is not None else df
    fig, axes = plt.subplots(1,2,figsize=(10,4))
    for ax, yvar, ylabel, ylim in zip(
        axes,
        ['co2',    'offset'],
        ['CO\u2082 (ppm)', 'CO\u2082 offset \u2014 Model minus Obs (ppm)'],
        [co2_ylim, offset_ylim]
    ):
        ydata = xdf['co2'] if yvar=='co2' else -xdf['offset']
        valid = xdf[np.isfinite(xdf[xvar]) & np.isfinite(ydata)]
        y     = valid['co2'] if yvar=='co2' else -valid['offset']
        ax.scatter(valid[xvar],y,color='#065A82',s=4,alpha=0.4,linewidths=0,zorder=2)
        m,b=np.polyfit(valid[xvar],y,1)
        x_fit=np.linspace(valid[xvar].min(),valid[xvar].max(),100)
        ax.plot(x_fit,m*x_fit+b,'r-',linewidth=2.0,label=f'slope={m:.3f}',zorder=3)
        if yvar=='offset': ax.axhline(0,color='black',linewidth=1.0,linestyle='--',zorder=1)
        ax.set_xlabel(xlabel,fontsize=11,fontweight='bold')
        ax.set_ylabel(ylabel,fontsize=11,fontweight='bold')
        ax.set_ylim(ylim)
        ax.legend(fontsize=10,framealpha=0.9)
        ax.grid(True,alpha=0.3)
    axes[0].set_title(f'CO\u2082 vs {xlabel.split(" (")[0]}',fontsize=12,fontweight='bold')
    axes[1].set_title(f'CO\u2082 Offset vs {xlabel.split(" (")[0]}',fontsize=12,fontweight='bold')
    fig.tight_layout()
    plt.savefig(FIG_DIR+save_prefix+'.png',dpi=150,bbox_inches='tight')
    plt.show()

scatter_two('sst',    'SST (\u00b0C)',         save_prefix='scatter_sst')
scatter_two('log_chl','log\u2081\u2080(Chl-a, mg m\u207b\u00b3)',
            xdf=df[np.isfinite(df['log_chl'])&(df['chl']>0)],
            save_prefix='scatter_chl')

# Pressure and wind side by side
fig, axes = plt.subplots(1,2,figsize=(10,4))
for ax, xvar, xlabel in zip(axes,['msl','wspd'],
                             ['MSL Pressure (hPa)','Surface Wind Speed (m/s)']):
    valid=df[np.isfinite(df[xvar])&np.isfinite(df['offset'])]
    ax.scatter(valid[xvar],-valid['offset'],color='#065A82',s=4,alpha=0.4,linewidths=0,zorder=2)
    m,b=np.polyfit(valid[xvar],-valid['offset'],1)
    x_fit=np.linspace(valid[xvar].min(),valid[xvar].max(),100)
    ax.plot(x_fit,m*x_fit+b,'r-',linewidth=2.0,label=f'slope={m:.3f}',zorder=3)
    ax.axhline(0,color='black',linewidth=1.0,linestyle='--',zorder=1)
    ax.set_xlabel(xlabel,fontsize=11,fontweight='bold')
    ax.set_ylabel('CO\u2082 offset \u2014 Model minus Obs (ppm)',fontsize=11,fontweight='bold')
    ax.set_ylim(-5,5)
    ax.legend(fontsize=10,framealpha=0.9)
    ax.grid(True,alpha=0.3)
axes[0].set_title('CO\u2082 Offset vs Pressure',fontsize=12,fontweight='bold')
axes[1].set_title('CO\u2082 Offset vs Wind Speed',fontsize=12,fontweight='bold')
fig.tight_layout()
plt.savefig(FIG_DIR+'scatter_era5.png',dpi=150,bbox_inches='tight')
plt.show()

## 13. GIF Animations
These cells generate animated GIFs of SST, chlorophyll, MSL pressure, and wind speed over the study period. Each frame is one monthly time step. Frames are saved to the frames directory then stitched into a GIF.

In [ ]:
def make_gif(time_arr, data_arr, lon_arr, lat_arr,
             cmap, vmin, vmax, cbar_label, title_prefix,
             frame_subdir, gif_name, norm=None):
    fdir = FRAME_DIR + frame_subdir + '/'
    os.makedirs(fdir, exist_ok=True)
    lon2d, lat2d = np.meshgrid(lon_arr, lat_arr)
    frames = []
    for i, t in enumerate(time_arr):
        fig = plt.figure(figsize=(12,3.5))
        ax  = fig.add_axes([0.05,0.1,0.82,0.82],
                            projection=ccrs.PlateCarree(central_longitude=180))
        ax.set_extent([-180,180,-23.5,23.5],crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND,facecolor='#D6CFC4',zorder=2)
        ax.add_feature(cfeature.COASTLINE,linewidth=0.5,zorder=3)
        gl=ax.gridlines(draw_labels=True,linewidth=0.4,color='grey',alpha=0.5,linestyle='--')
        gl.top_labels=False; gl.right_labels=False
        d = data_arr[i].copy()
        if norm:
            pc=ax.pcolormesh(lon2d,lat2d,d,cmap=cmap,norm=norm,
                             transform=ccrs.PlateCarree(),zorder=1)
        else:
            pc=ax.pcolormesh(lon2d,lat2d,d,cmap=cmap,vmin=vmin,vmax=vmax,
                             transform=ccrs.PlateCarree(),zorder=1)
        cbar_ax=fig.add_axes([0.89,0.15,0.02,0.7])
        plt.colorbar(pc,cax=cbar_ax).set_label(cbar_label,fontsize=10)
        tstr = t.strftime('%B %Y') if hasattr(t,'strftime') else str(t)
        ax.set_title(f'{title_prefix}  |  {tstr}',fontsize=13,fontweight='bold')
        fname=fdir+f'frame_{i:03d}.png'
        plt.savefig(fname,dpi=100,facecolor='white')
        frames.append(fname)
        plt.close(fig)
        print(f'  Frame {i+1}/{len(time_arr)}: {tstr}')
    gif_path=GIF_DIR+gif_name
    with imageio.get_writer(gif_path,mode='I',duration=500,loop=0) as writer:
        for fname in frames: writer.append_data(imageio.imread(fname))
    print(f'GIF saved to {gif_path}')

# SST
make_gif(sst_time, sst_data, sst_lon, sst_lat,
         'RdYlBu_r', 20, 32, 'SST (\u00b0C)', 'OISST Sea Surface Temperature',
         'sst', 'sst_tropics_2016_2018.gif')

# Chlorophyll
chl_data_clean = chl_data.copy()
chl_data_clean[chl_data_clean<=0] = np.nan
make_gif(chl_time, chl_data_clean, chl_lon, chl_lat,
         'YlGn', None, None, 'Chl-a (mg m\u207b\u00b3)', 'MODIS Chlorophyll-a',
         'chl', 'chl_tropics_2016_2018.gif',
         norm=mcolors.LogNorm(vmin=0.01,vmax=1.0))

# MSL pressure
make_gif(era5_time, msl[:,:,sort_idx], era5_lon_sorted, era5_lat,
         'RdBu_r', 1005, 1025, 'MSL Pressure (hPa)', 'ERA5 Mean Sea Level Pressure',
         'msl', 'msl_tropics_2016_2018.gif')

# Wind speed
make_gif(era5_time, wspd[:,:,sort_idx], era5_lon_sorted, era5_lat,
         'Blues', 0, 12, 'Wind Speed (m/s)', 'ERA5 10m Wind Speed',
         'wind', 'wind_tropics_2016_2018.gif')

## 14. Random Forest — Predicting CO₂
This cell trains a random forest regressor to predict CO2 from 7 environmental and geographic features. We use 200 trees with max depth 6 and min leaf size 20 to prevent overfitting. The 80/20 train/test split gives an honest evaluation on data the model never saw during training.

In [ ]:
features = ['sst','log_chl','wspd','msl','lat','lon','doy']
target   = 'co2'

df_ml = df[features+[target]].dropna().copy()
X     = df_ml[features].values
y     = df_ml[target].values
print(f'Training samples: {len(X)}')

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

rf = RandomForestRegressor(n_estimators=200,max_depth=6,
                            min_samples_leaf=20,random_state=42,n_jobs=-1)
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
r2  = r2_score(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
print(f'R\u00b2  = {r2:.3f}')
print(f'MAE = {mae:.3f} ppm')

feature_labels = {'sst':'SST (\u00b0C)','log_chl':'log\u2081\u2080(Chl-a)',
                  'wspd':'Wind Speed','msl':'MSL Pressure',
                  'lat':'Latitude','lon':'Longitude','doy':'Day of Year'}
importances = rf.feature_importances_
labels      = [feature_labels[f] for f in features]
sort_idx    = np.argsort(importances)[::-1]

# PLOT: Feature importances
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(range(len(features)),importances[sort_idx],color='#2A7AE0',alpha=0.85,edgecolor='white')
ax.set_xticks(range(len(features)))
ax.set_xticklabels([labels[i] for i in sort_idx],fontsize=11)
ax.set_ylabel('Feature Importance (Gini)',fontsize=12)
ax.set_title(f'Random Forest Feature Importances\nR\u00b2 = {r2:.3f} | MAE = {mae:.3f} ppm',
             fontsize=13,fontweight='bold')
ax.grid(True,alpha=0.3,axis='y')
fig.tight_layout()
plt.savefig(FIG_DIR+'rf_feature_importance.png',dpi=150,bbox_inches='tight')
plt.show()

# PLOT: Predicted vs actual
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(y_test,y_pred,s=6,alpha=0.4,color='#065A82',linewidths=0,zorder=2)
ax.plot([395,420],[395,420],'r-',linewidth=2.0,label='1:1 line',zorder=3)
ax.set_xlim(395,420); ax.set_ylim(395,420)
ax.set_xlabel('Actual CO\u2082 (ppm)',fontsize=12,fontweight='bold')
ax.set_ylabel('Predicted CO\u2082 (ppm)',fontsize=12,fontweight='bold')
ax.set_title(f'Random Forest \u2014 Predicted vs Actual CO\u2082\nR\u00b2 = {r2:.3f}  |  MAE = {mae:.3f} ppm',
             fontsize=12,fontweight='bold')
ax.legend(fontsize=11,framealpha=0.9)
ax.grid(True,alpha=0.3)
ax.set_aspect('equal')
fig.tight_layout()
plt.savefig(FIG_DIR+'rf_predicted_vs_actual.png',dpi=150,bbox_inches='tight')
plt.show()

# PLOT: Partial dependence for top 4 features
top4 = list(sort_idx[:4])
fig, axes = plt.subplots(2,2,figsize=(12,8))
PartialDependenceDisplay.from_estimator(
    rf, X_train, top4, feature_names=features,
    ax=axes.flatten(), line_kw={'color':'#2A7AE0','linewidth':2}
)
for ax, idx in zip(axes.flatten(),top4):
    ax.set_xlabel(labels[idx],fontsize=11)
    ax.set_ylabel('Partial dependence\n(CO\u2082, ppm)',fontsize=10)
    ax.grid(True,alpha=0.3)
fig.suptitle('Partial Dependence Plots \u2014 Top 4 Features',fontsize=13,fontweight='bold')
fig.tight_layout()
plt.savefig(FIG_DIR+'rf_partial_dependence.png',dpi=150,bbox_inches='tight')
plt.show()